# Train the modified main model from scratch

Select the RTX PRO 6000 GPU in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated"
DATA = WORKSPACE / "data" / "fineweb_edu"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([sys.executable, "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)

# Training runs in a subprocess of the same Python; tqdm renders in the notebook.
sys.path.insert(0, str(REPO / "src"))
from base_model_research.progress import run_with_progress

def training_data(*, local_copy=True):
    if not (SAVED_DATA / "manifest.json").is_file():
        raise FileNotFoundError(f"Run notebook 01 to prepare the shared dataset at {SAVED_DATA}.")
    if not local_copy or SAVED_DATA == DATA:
        return ["--data", SAVED_DATA]
    if not (DATA / "manifest.json").is_file():
        from inspect import signature
        from base_model_research.data import TokenCorpus
        if "cache_dir" in signature(TokenCorpus).parameters:
            return ["--data", SAVED_DATA, "--cache-dir", DATA.parent / "cached_shards"]
    # Reuse the prepared SSD corpus, or stage it for an older pinned trainer.
    run("copy-data", "--source", SAVED_DATA, "--destination", DATA)
    return ["--data", DATA]

def run_training(command_name, model_name, run_record, *arguments):
    # Existing runs keep their settings. New runs use the profile from 00 or the training config.
    settings_file = run_record if run_record.is_file() else RUNS / f"{model_name}_profile.json"
    memory_options = []
    if settings_file.is_file():
        settings = json.loads(settings_file.read_text())["training"]
        memory_options = ["--microbatch-size", str(settings["microbatch_size"])]
        if settings["activation_checkpointing"]:
            memory_options.append("--activation-checkpointing")
    command = [sys.executable, "-u", "-m", "base_model_research.cli", command_name,
               "--model", f"configs/models/{model_name}.yaml", *map(str, arguments),
               *memory_options, "--progress-events"]
    run_with_progress(command, cwd=REPO)


Trains once through this family's token budget and validates at 5% intervals. Each of the 20 validations saves the model weights and optimizer state to Drive. The 20th save includes the final optimizer update. Rerun this notebook to continue from the latest completed save. Allow about 144 GB on Drive per main model.


In [ ]:
OUTPUT = RUNS / "main" / "modified"
WINNER = RUNS / "proxy" / "modified" / "best_hparams.json"
if not WINNER.is_file():
    raise FileNotFoundError("Enable RUN_FULL in notebook 03 "
                            f"and finish the modified proxy search to create {WINNER}.")
options = ["--resume"] if (OUTPUT / "run.json").exists() else []
run_training("train", "modified_main", OUTPUT / "run.json", "--hyperparameters", WINNER,
    *training_data(), "--output", OUTPUT, *options)
